# 02_tracts_and_strata

Tract-set consistency across files + strata profile of the scored tracts per region.

*Converted from the former `analysis_scripts/02_tracts_and_strata.py`.*

In [1]:
import pathlib, zipfile
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
DATA = ROOT / "data"


def P(rel):
    """Path to a data file: use the extracted copy if present, otherwise stream it straight out of the zip."""
    rel = str(rel)
    if (DATA / rel).exists():
        return DATA / rel
    for zname in ("reference.zip", "strata.zip"):
        z = zipfile.ZipFile(DATA / zname)
        if rel in z.namelist():
            return z.open(rel)          # stored (uncompressed) members are seekable, so pyarrow reads them in place
    raise FileNotFoundError(rel)

In [2]:
import pathlib

import numpy as np
import pandas as pd

ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
DATA = ROOT / "data"
OUT = ROOT / "analysis_outputs"
REGIONS = ["maricopa-az", "northern-ca", "eastern-ok", "south-central-tx", "eastern-wa"]
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 500)

lines = []


def log(*a):
    s = " ".join(str(x) for x in a)
    print(s)
    lines.append(s)

In [3]:
# ---------- A. tract-set consistency ----------

In [4]:
log("=" * 30, "A. TRACT SET CONSISTENCY", "=" * 30)
sets = {}
for r in REGIONS:
    sub = pd.read_csv(P(f"reference/{r}/{r}-sample-submission.csv"), dtype={"GEOID": str})
    bnd = pd.read_csv(P(f"boundaries/{r}-tract-geoids.csv"), dtype={"GEOID": str})
    trc = pd.read_parquet(P(f"strata/{r}/{r}-census-tracts.parquet"), columns=["GEOID", "frac_inside_aoi"])
    acs = pd.read_parquet(P(f"reference/{r}/{r}-census-acs-housing.parquet"), columns=["GEOID", "housing_units"])
    stt = pd.read_parquet(P(f"strata/{r}/{r}-strata-tract-table.parquet"), columns=["GEOID"])
    s_sub, s_bnd, s_trc, s_acs, s_stt = (set(x.GEOID) for x in (sub, bnd, trc, acs, stt))
    sets[r] = s_sub
    log(f"\n[{r}] sub={len(sub)} (unique {len(s_sub)}) boundaries={len(s_bnd)} census-tracts={len(s_trc)} "
        f"acs={len(s_acs)} strata={len(s_stt)}")
    log(f"  GEOID lengths in sub: {sub.GEOID.str.len().value_counts().to_dict()}  "
        f"state FIPS: {sub.GEOID.str[:2].value_counts().to_dict()}  counties: {sub.GEOID.str[:5].nunique()}")
    log(f"  sub columns: {list(sub.columns)}; all-zero values: {bool((sub.drop(columns='GEOID') == 0).all().all())}")
    log(f"  boundaries - sub: {sorted(s_bnd - s_sub)}")
    log(f"  sub - boundaries: {sorted(s_sub - s_bnd)};  sub == census-tracts: {s_sub == s_trc};  "
        f"sub == strata: {s_sub == s_stt};  acs - sub: {sorted(s_acs - s_sub)}; sub - acs: {sorted(s_sub - s_acs)}")
    log(f"  frac_inside_aoi range: {trc.frac_inside_aoi.min()} .. {trc.frac_inside_aoi.max()}")
all_ids = [g for s in sets.values() for g in s]
log(f"\nTotal scored tracts (5 regions): {len(all_ids)}; unique: {len(set(all_ids))}; "
    f"4 Zindi regions (excl. eastern-wa): {sum(len(sets[r]) for r in REGIONS if r != 'eastern-wa')}")

============================== A. TRACT SET CONSISTENCY ==============================

[maricopa-az] sub=1593 (unique 1593) boundaries=1593 census-tracts=1593 acs=1593 strata=1593
  GEOID lengths in sub: {11: 1593}  state FIPS: {'04': 1592, '35': 1}  counties: 16
  sub columns: ['GEOID', 'transport_gap', 'building_gap', 'poi_gap', 'coverage_gap_score']; all-zero values: True
  boundaries - sub: []
  sub - boundaries: [];  sub == census-tracts: True;  sub == strata: True;  acs - sub: []; sub - acs: []
  frac_inside_aoi range: 1.0 .. 1.0

[northern-ca] sub=591 (unique 591) boundaries=591 census-tracts=591 acs=591 strata=591
  GEOID lengths in sub: {11: 591}  state FIPS: {'06': 591}  counties: 24
  sub columns: ['GEOID', 'transport_gap', 'building_gap', 'poi_gap', 'coverage_gap_score']; all-zero values: True
  boundaries - sub: []
  sub - boundaries: [];  sub == census-tracts: True;  sub == strata: True;  acs - sub: []; sub - acs: []
  frac_inside_aoi range: 1.0 .. 1.0

[eastern-ok] sub=


[south-central-tx] sub=6003 (unique 6003) boundaries=6010 census-tracts=6010 acs=6010 strata=6010
  GEOID lengths in sub: {11: 6003}  state FIPS: {'48': 6003}  counties: 185
  sub columns: ['GEOID', 'transport_gap', 'building_gap', 'poi_gap', 'coverage_gap_score']; all-zero values: True
  boundaries - sub: ['48007990000', '48061990000', '48261990000', '48273990000', '48321990000', '48355990000', '48489990000']
  sub - boundaries: [];  sub == census-tracts: False;  sub == strata: False;  acs - sub: ['48007990000', '48061990000', '48261990000', '48273990000', '48321990000', '48355990000', '48489990000']; sub - acs: []
  frac_inside_aoi range: 1.0 .. 1.0

[eastern-wa] sub=415 (unique 415) boundaries=415 census-tracts=415 acs=415 strata=415
  GEOID lengths in sub: {11: 415}  state FIPS: {'53': 415}  counties: 27
  sub columns: ['GEOID', 'transport_gap', 'building_gap', 'poi_gap', 'coverage_gap_score']; all-zero values: True
  boundaries - sub: []
  sub - boundaries: [];  sub == census-tra

In [5]:
# ---------- B. strata profile ----------
log("\n" + "=" * 30, "B. STRATA PROFILE (scored tracts, national strata table)", "=" * 30)
nat = pd.read_parquet(P("strata/national/national-strata-tract-table.parquet"))
reg_of = {g: r for r, s in sets.items() for g in s}
df = nat[nat.GEOID.isin(reg_of)].copy()
df["region"] = df.GEOID.map(reg_of)
df.to_parquet(OUT / "scored_tracts_strata.parquet")
log(f"rows matched: {len(df)} / {len(reg_of)}")

df["water_share"] = df.AWATER / (df.ALAND + df.AWATER)
df["aland_km2"] = df.ALAND / 1e6
df["pop_density_km2"] = df.pop_total / df.aland_km2.replace(0, np.nan)


def q(s):
    s = s.dropna()
    if s.empty:
        return "all null"
    return (f"n={len(s)} min={s.min():.3g} p10={s.quantile(.1):.3g} p50={s.median():.3g} "
            f"mean={s.mean():.3g} p90={s.quantile(.9):.3g} max={s.max():.3g}")


g = df.groupby("region")
log("\n-- Size / population --")
for r, d in g:
    log(f"[{r}] tracts={len(d)} pop_total_sum={d.pop_total.sum():,} zero-pop tracts={int((d.pop_total == 0).sum())} "
        f"pop<100={int((d.pop_total < 100).sum())}")
    log(f"    pop_total: {q(d.pop_total)}")
    log(f"    ALAND km2: {q(d.aland_km2)}")
    log(f"    pop density /km2: {q(d.pop_density_km2)}")
    log(f"    water_share: {q(d.water_share)}  tracts water_share>0.5: {int((d.water_share > .5).sum())}")

log("\n-- Urban/rural (ur_class, census 2020) --")
log(pd.crosstab(df.region, df.ur_class, margins=True).to_string())
log("\npct_urban buckets:")
log(pd.crosstab(df.region, pd.cut(df.pct_urban, [-0.01, 0, 0.5, 0.99, 1.0],
                                  labels=["0", "(0,.5]", "(.5,.99]", "1"])).to_string())
log("\n-- RUCA primary (1=metro core .. 10=rural) --")
log(pd.crosstab(df.region, df.ruca_primary.fillna(-1).astype(int), margins=True).to_string())
log("\n-- RUCC 2023 (county) --")
log(pd.crosstab(df.region, df.rucc_2023.fillna(-1).astype(int)).to_string())
log("\n-- NCHS 2013 (1=large central metro .. 6=noncore) --")
log(pd.crosstab(df.region, df.nchs_2013.fillna(-1).astype(int)).to_string())

log("\n-- Tribal --")
log(pd.crosstab(df.region, df.tribal_any, margins=True).to_string())
log(pd.crosstab(df.region, [df.tribal_legal, df.tribal_statistical]).to_string())
for r, d in g:
    t = d[d.tribal_any == True]
    log(f"[{r}] tribal tracts={len(t)} tribal_pct: {q(t.tribal_pct)}")
    log(f"    aiannh_kind: {t.aiannh_kind.value_counts().to_dict()}")
    log(f"    top aiannh: {t.aiannh_name.value_counts().head(8).to_dict()}")

log("\n-- SVI / CVI --")
for r, d in g:
    log(f"[{r}] svi_covered={d.svi_covered.mean():.3f} svi_overall: {q(d.svi_overall)}  svi<0 count={int((d.svi_overall < 0).sum())}")
    log(f"    cvi_covered={d.cvi_covered.mean():.3f} cvi_overall: {q(d.cvi_overall)}")

log("\n-- Hazards (headline) --")
haz = ["usdm_summer_dsci", "usdm_winter_dsci", "usfs_WHP_mean", "usfs_RPS_mean", "usfs_BP_mean",
       "epht_heat_days_summer", "gehe_wbgt28_days_mean", "hwd_heatindex_mean", "mtbs_wildfire_burned_pct_land",
       "nifc_wildfire_ever", "fod_fires", "carbonplan_rps_2011_mean", "pmdi_mean", "spi12_mean"]
for c in haz:
    log(f"{c}:")
    for r, d in g:
        s = d[c].astype(float) if d[c].dtype == bool else d[c]
        log(f"   [{r:<16}] {q(s)}")

log("\n-- *_covered flags (share True) --")
cov = [c for c in df.columns if c.endswith("_covered")]
log(df.groupby("region")[cov].mean().round(3).T.to_string())

log("\n-- Missingness: columns with any nulls, share null per region --")
na = df.drop(columns=["region"]).isna().groupby(df.region).mean()
na = na.loc[:, na.max() > 0].T.round(3)
log(f"{len(na)} of {df.shape[1]-1} columns have nulls somewhere")
log(na.to_string())

log("\n-- Independent building / housing counts carried in strata --")
for c in ["usfs_BuildingCount_sum", "usfs_HUCount_sum", "usfs_PopCount_sum", "carbonplan_buildings",
          "usfs_BuildingDensity_mean", "usfs_BuildingCover_mean"]:
    log(f"{c}:")
    for r, d in g:
        log(f"   [{r:<16}] {q(d[c])}  nulls={int(d[c].isna().sum())}")

(OUT / "02_tracts_and_strata.txt").write_text("\n".join(lines), encoding="utf-8")


============================== B. STRATA PROFILE (scored tracts, national strata table) ==============================


rows matched: 9794 / 9794

-- Size / population --
[eastern-ok] tracts=1192 pop_total_sum=3,926,466 zero-pop tracts=1 pop<100=8
    pop_total: n=1192 min=0 p10=1.63e+03 p50=3.13e+03 mean=3.29e+03 p90=5.17e+03 max=1.11e+04
    ALAND km2: n=1192 min=0.122 p10=1.33 p50=10.2 mean=135 p90=457 max=2.96e+03
    pop density /km2: n=1192 min=0 p10=6.21 p50=328 mean=662 p90=1.72e+03 max=4.63e+03
    water_share: n=1192 min=0 p10=0 p50=0.00214 mean=0.0196 p90=0.0465 max=0.603  tracts water_share>0.5: 1
[eastern-wa] tracts=415 pop_total_sum=1,676,917 zero-pop tracts=1 pop<100=2
    pop_total: n=415 min=0 p10=2.15e+03 p50=3.78e+03 mean=4.04e+03 p90=6.34e+03 max=1.06e+04
    ALAND km2: n=415 min=0.575 p10=1.55 p50=10.7 mean=311 p90=1.12e+03 max=4.62e+03
    pop density /km2: n=415 min=0 p10=3.01 p50=392 mean=817 p90=2.21e+03 max=4.82e+03
    water_share: n=415 min=0 p10=0 p50=0.000611 mean=0.0222 p90=0.0509 max=0.399  tracts water_share>0.5: 0
[maricopa-az] tracts=1593 pop_total_sum=6,606,494 zero-p

   [south-central-tx] n=6003 min=0 p10=1.07 p50=18.8 mean=36.9 p90=94 max=912
usfs_RPS_mean:
   [eastern-ok      ] n=1192 min=0 p10=9.97e-05 p50=0.0232 mean=0.0659 p90=0.2 max=0.662
   [eastern-wa      ] n=415 min=0 p10=0.000106 p50=0.0246 mean=0.2 p90=0.593 max=3.82
   [maricopa-az     ] n=1593 min=0 p10=5.37e-06 p50=0.00134 mean=0.0506 p90=0.124 max=1.88
   [northern-ca     ] n=591 min=0 p10=0.000121 p50=0.0155 mean=0.179 p90=0.634 max=2.16
   [south-central-tx] n=6003 min=0 p10=3.46e-05 p50=0.00283 mean=0.0182 p90=0.0445 max=0.635
usfs_BP_mean:
   [eastern-ok      ] n=1192 min=0 p10=5.96e-06 p50=0.000789 mean=0.00212 p90=0.00649 max=0.0159
   [eastern-wa      ] n=415 min=0 p10=4.12e-06 p50=0.000709 mean=0.00498 p90=0.0157 max=0.0653
   [maricopa-az     ] n=1593 min=0 p10=1.74e-07 p50=4.42e-05 mean=0.00114 p90=0.00299 max=0.0287
   [northern-ca     ] n=591 min=0 p10=6.28e-06 p50=0.000557 mean=0.00363 p90=0.0125 max=0.0341
   [south-central-tx] n=6003 min=0 p10=1.6e-06 p50=0.000107 me

   [maricopa-az     ] n=1593 min=2.47 p10=813 p50=1.57e+03 mean=1.72e+03 p90=2.72e+03 max=2.86e+04  nulls=0
   [northern-ca     ] n=591 min=150 p10=839 p50=1.77e+03 mean=1.89e+03 p90=3.1e+03 max=5.16e+03  nulls=0
   [south-central-tx] n=6003 min=0 p10=564 p50=1.55e+03 mean=1.75e+03 p90=3.18e+03 max=1.02e+04  nulls=0
usfs_HUCount_sum:
   [eastern-ok      ] n=1192 min=0 p10=759 p50=1.4e+03 mean=1.46e+03 p90=2.25e+03 max=4.36e+03  nulls=0
   [eastern-wa      ] n=415 min=0.858 p10=873 p50=1.63e+03 mean=1.67e+03 p90=2.49e+03 max=3.78e+03  nulls=0
   [maricopa-az     ] n=1593 min=0 p10=849 p50=1.74e+03 mean=1.77e+03 p90=2.71e+03 max=7.63e+03  nulls=0
   [northern-ca     ] n=591 min=37.1 p10=938 p50=1.69e+03 mean=1.7e+03 p90=2.52e+03 max=3.76e+03  nulls=0
   [south-central-tx] n=6003 min=0 p10=918 p50=1.61e+03 mean=1.7e+03 p90=2.58e+03 max=1e+04  nulls=0
usfs_PopCount_sum:
   [eastern-ok      ] n=1192 min=0 p10=1.78e+03 p50=3.42e+03 mean=3.56e+03 p90=5.58e+03 max=1.23e+04  nulls=0
   [eastern

30558